# 🚀 DecodeLabs — AI Project 3
## Tech Stack Recommender
**Algorithm:** TF-IDF Vectorization + Cosine Similarity  
**Pipeline:** Ingestion → Scoring → Sorting → Filtering

## Step 0: Install & Import Libraries

In [ ]:
!pip install pandas scikit-learn -q

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('✅ Libraries imported successfully')

## Step 1: Create the Dataset
Each row = one job role with its required skills (space-separated).  
These are the **"items"** our recommendation engine will match against.

In [ ]:
data = {
    'role': [
        'Frontend Developer',
        'Backend Developer',
        'Full Stack Developer',
        'Data Scientist',
        'DevOps Engineer',
        'Cloud Architect',
        'ML Engineer',
        'Mobile Developer',
        'Cybersecurity Analyst',
        'Data Analyst',
        'System Administrator',
        'Database Administrator',
        'AI Engineer',
        'Blockchain Developer',
        'Embedded Systems Engineer',
        'Game Developer',
        'QA Engineer',
        'NLP Engineer',
        'Computer Vision Engineer',
        'Site Reliability Engineer'
    ],
    'skills': [
        'html css javascript react typescript redux figma ui design responsive animations',
        'python java nodejs sql rest api postgresql mongodb express spring boot authentication',
        'html css javascript react nodejs python sql mongodb rest api git deployment',
        'python machine learning sql statistics data analysis tensorflow pandas numpy scipy visualization',
        'aws docker kubernetes cicd linux git terraform jenkins shell scripting automation',
        'aws azure gcp cloud computing docker kubernetes networking security infrastructure iac',
        'python machine learning tensorflow pytorch deep learning neural networks optimization algorithms',
        'flutter react native android ios java swift kotlin firebase push notifications',
        'security networking linux python encryption ethical hacking firewalls vulnerability penetration',
        'sql excel power bi tableau data analysis statistics python reporting dashboards',
        'linux networking windows server shell scripting automation bash it support monitoring',
        'sql postgresql mysql mongodb oracle database design query optimization indexing backup',
        'python machine learning deep learning nlp computer vision tensorflow pytorch langchain openai',
        'solidity web3 ethereum smart contracts cryptography javascript typescript dapp',
        'c cpp arduino raspberry pi microcontrollers iot hardware firmware sensors',
        'unity c++ csharp unreal engine opengl physics animation 3d modeling shaders',
        'testing automation selenium pytest junit bug tracking ci pipeline quality assurance',
        'python nlp text processing transformers bert spacy nltk sentiment classification',
        'python opencv deep learning image processing tensorflow pytorch cnn object detection',
        'linux cloud monitoring sre kubernetes prometheus grafana incident response automation'
    ]
}

df = pd.DataFrame(data)
df.to_csv('raw_skills.csv', index=False)

print(f'✅ Dataset created: {len(df)} job roles')
df

## Step 2: Build the TF-IDF Model
**TF (Term Frequency):** How often a skill appears in a job role description.  
**IDF (Inverse Document Frequency):** Penalizes common skills (like `python`) and rewards rare/specific ones (like `pytorch`).  
Each job role becomes a **weighted numerical vector** in the same vocabulary space.

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['skills'])

print(f'✅ TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'   → {tfidf_matrix.shape[0]} job roles × {tfidf_matrix.shape[1]} unique skill terms')

### 🔍 Peek: IDF Weights of Some Skills
Higher IDF = more specific/rare skill → carries more weight in similarity matching.

In [ ]:
feature_names = vectorizer.get_feature_names_out()
idf_scores = vectorizer.idf_

idf_df = pd.DataFrame({'skill': feature_names, 'idf_weight': idf_scores})
idf_df = idf_df.sort_values('idf_weight', ascending=False)

print('Top 10 most SPECIFIC skills (high IDF):')
print(idf_df.head(10).to_string(index=False))
print()
print('Top 10 most COMMON skills (low IDF):')
print(idf_df.tail(10).to_string(index=False))

## Step 3: Take User Input (Ingestion)
Minimum **3 skills** required for sufficient data density in matching.

In [ ]:
def get_user_input():
    """Prompt user for skills, enforce minimum 3."""
    while True:
        raw = input('Enter your skills (comma-separated, min 3): ').strip()
        skills = [s.strip().lower() for s in raw.split(',') if s.strip()]
        if len(skills) >= 3:
            print(f'\n✅ Skills captured: {skills}')
            return ' '.join(skills)   # Single string for TF-IDF
        print('⚠  Please enter at least 3 skills.\n')

user_skills = get_user_input()

## Step 4: Score — Cosine Similarity
Transform user input into the **same TF-IDF vector space**, then measure the **angular alignment** (cosine) between the user vector and each job role vector.  

- Score **1.0** → Perfect match  
- Score **0.0** → No common skills

In [ ]:
# Transform user input into TF-IDF vector
user_vector = vectorizer.transform([user_skills])

# Compute cosine similarity against all job roles
scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

# Attach scores to dataframe
df['match_score'] = scores

print('✅ Cosine similarity scores computed for all job roles:')
print(df[['role', 'match_score']].sort_values('match_score', ascending=False).to_string(index=False))

## Step 5: Sort & Filter — Top 3 Recommendations

In [ ]:
# Sort descending, keep top 3
top3 = df.sort_values('match_score', ascending=False).head(3).reset_index(drop=True)

print('=' * 50)
print('   🎯  TOP 3 CAREER PATH RECOMMENDATIONS')
print('=' * 50)

if top3['match_score'].max() == 0:
    print('No matches found. Try different or more specific skills.')
else:
    for rank, row in top3.iterrows():
        bar = '█' * int(row['match_score'] * 25)
        print(f"\n  #{rank+1}  {row['role']}")
        print(f"       Match : {row['match_score']:.2%}  {bar}")

print('\n' + '=' * 50)

## Step 6: Full Pipeline — Reusable Function
Wrapping everything into one clean `recommend()` function.

In [ ]:
def recommend(skills_input, top_n=3):
    """
    Full recommendation pipeline.
    Input  : comma-separated skills string
    Output : Top-N matching job roles with scores
    """
    # Clean input
    skills = ' '.join([s.strip().lower() for s in skills_input.split(',')])

    # Vectorize user input
    user_vec = vectorizer.transform([skills])

    # Score
    scores = cosine_similarity(user_vec, tfidf_matrix).flatten()

    # Sort & Filter
    result = df.copy()
    result['match_score'] = scores
    top = result.sort_values('match_score', ascending=False).head(top_n).reset_index(drop=True)

    # Display
    print(f'\nSkills entered : {skills_input}')
    print('-' * 40)
    for rank, row in top.iterrows():
        bar = '█' * int(row['match_score'] * 25)
        print(f"#{rank+1}  {row['role']:<28} {row['match_score']:.2%}  {bar}")


# ── Test Cases ───────────────────────────────────────────────
recommend('Python, Machine Learning, TensorFlow')
recommend('AWS, Docker, Kubernetes, Linux')
recommend('HTML, CSS, React, JavaScript')
recommend('SQL, Excel, Tableau, Statistics')

## Step 7: Visualize — Bar Chart of Match Scores

In [ ]:
import matplotlib.pyplot as plt

def plot_recommendations(skills_input, top_n=5):
    skills = ' '.join([s.strip().lower() for s in skills_input.split(',')])
    user_vec = vectorizer.transform([skills])
    scores = cosine_similarity(user_vec, tfidf_matrix).flatten()

    result = df.copy()
    result['match_score'] = scores
    top = result.sort_values('match_score', ascending=False).head(top_n)

    colors = ['#2ecc71' if i == 0 else '#3498db' if i == 1 else '#e67e22' if i == 2 else '#95a5a6'
              for i in range(top_n)]

    plt.figure(figsize=(9, 4))
    bars = plt.barh(top['role'][::-1], top['match_score'][::-1], color=colors[::-1])
    plt.xlabel('Cosine Similarity Score')
    plt.title(f'Top {top_n} Career Matches for: "{skills_input}"')
    plt.xlim(0, 1)

    for bar, score in zip(bars, top['match_score'][::-1]):
        plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{score:.2%}', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()


plot_recommendations('Python, Machine Learning, TensorFlow')

---
## ✅ Summary

| Pipeline Step | What Happens |
|---|---|
| **Ingestion** | User enters ≥3 skills → cleaned & lowercased |
| **Vectorization** | TF-IDF converts skills into weighted numerical vectors |
| **Scoring** | Cosine Similarity computed between user vector & all job roles |
| **Sorting** | Results ranked highest score first |
| **Filtering** | Only Top-3 returned (avoids choice overload) |

> **Why Cosine over Euclidean?** Cosine measures *direction* (what skills you have), not *magnitude* (how many). A user with 3 skills vs 10 skills pointing in the same direction gets the same score — which is the correct behavior.